In [3]:
pip install --upgrade google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.4/791.4 kB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 21.8 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [cffi]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [google-genai] [google-genai]s]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
api_key = os.environ.get("GEMINI_API_KEY")

In [19]:
import os
import sys
import time
import json
import random
import re
import pandas as pd
from google import genai
from google.genai import types

# ==========================================
# [설정값]
# ==========================================
# api_key 변수는 이미 선언되어 있다고 가정합니다. (예: api_key = "AIzaSy...")
client = genai.Client(api_key=api_key)
MODEL_NAME = 'gemma-4-31b-it'

INPUT_FILE = 'filtered_concat_translated_problems.csv'
OUTPUT_FILE = 'problem_algorithms.csv'

# [분산 처리용 범위 설정] (테이블 기준 상단 첫 번째 = 1)
START_IDX = 1       # 시작 문제 순서
END_IDX = 360      # 끝 문제 순서

# 대상 알고리즘 리스트 (수정됨)
ALGORITHMS = [
    "BFS",
    "Backtracking",
    "Binary Search",
    "Bitmasking",
    "Brute Force",
    "DFS",
    "Dijkstra",
    "Dynamic Programming",
    "Floyd-Warshall",
    "Greedy",
    "Hash Table",
    "Heap",
    "Implementation",
    "KMP",
    "Kruskal",
    "LCA",
    "Linked List",
    "Parametric Search",
    "Prim",
    "Priority Queue",
    "Segment Tree",
    "Sliding Window",
    "Topological Sort",
    "Trie",
    "Two Pointers",
    "Union-Find"
]

# ==========================================
# [함수 정의] API 호출 및 JSON 파싱
# ==========================================
def classify_algorithms_with_llm(problem_text, max_retries=3):
    """
    LLM에 영문 프롬프트를 보내어 알고리즘 조합을 JSON 형태로 받아옵니다.
    """
    algorithms_str = ", ".join([f'"{algo}"' for algo in ALGORITHMS])
    
    prompt = f"""
You are an expert competitive programming coach. 
Read the following problem description carefully.

[Problem Context]
{problem_text}

Your task is to identify the most common and standard combinations of algorithms that can optimally solve this problem.
A problem might be solved using just one core algorithm (e.g., BFS) or a combination of two core algorithms (e.g., Dynamic Programming + Bitmasking). 
There could be multiple distinct standard approaches to solve the same problem.

[CRITICAL RULES]
1. ALGORITHM LIST: You MUST select algorithms ONLY from the following exact list. DO NOT use any algorithm names outside this list:
[{algorithms_str}]

2. LIMIT OF 2 ALGORITHMS PER COMBINATION: For each approach (combination), you must select AT MOST 2 algorithms. Focus strictly on the most CORE algorithms needed to solve the problem. Do not list minor or trivial steps.

3. STANDARD APPROACHES ONLY: Provide generally expected and standard algorithm combinations. Avoid overly unusual, complex, or niche approaches.

4. DEFINITION OF "Implementation": In this task, "Implementation" strictly refers to "State Simulation" types (e.g., directional movement, state transitions, turn-based progression, 2D grid simulation). DO NOT select "Implementation" for mere simple coding tasks or basic logic evaluation.

5. JSON FORMAT: Return the result STRICTLY as a JSON array of objects. Each object represents ONE valid combination of algorithms. Inside each object, include ONLY the algorithms used in that specific combination, and set their values to true.

[Example Output Format]
If the problem is standardly solved by either (Approach 1: BFS alone) OR (Approach 2: DFS combined with Dijkstra), return:
[
  {{ "BFS": true }},
  {{ "DFS": true, "Dijkstra": true }}
]

Provide only the pure JSON array. No markdown tags, no explanations.
"""

    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.4 # 논리적 분석이 필요하므로 약간 낮은 온도 설정
    )
    
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=config
            )
            
            text = response.text.strip()
            
            # 마크다운 찌꺼기 방어
            md_marker = "`" * 3
            if text.startswith(f'{md_marker}json'):
                text = text.replace(f'{md_marker}json', '').replace(md_marker, '').strip()
            elif text.startswith(md_marker):
                text = text.replace(md_marker, '').strip()

            parsed_json = json.loads(text)
            
            if isinstance(parsed_json, list) and len(parsed_json) > 0:
                return parsed_json
            else:
                raise ValueError("JSON is not a valid list or is empty.")
                
        except Exception as e:
            if attempt == max_retries:
                print(f"        ❌ [LLM 실패] 최대 재시도 횟수 초과. 에러: {e}")
                return None
                
            sleep_time = (10 * attempt) + random.uniform(1, 3)
            print(f"        ⚠️ API 에러({e}). {sleep_time:.1f}초 후 재시도합니다... ({attempt}/{max_retries})")
            time.sleep(sleep_time)

# ==========================================
# [메인 파이프라인 시작]
# ==========================================
print("🚀 [Step 1] 데이터 로드 및 초기화...")
df_in = pd.read_csv(INPUT_FILE)

# 결과 저장용 CSV 구조 셋업 및 이미 처리된 문제 ID 스캔
processed_ids = set()
csv_columns = ['problem_id', 'algorithm_order', 'count'] + ALGORITHMS

if os.path.exists(OUTPUT_FILE):
    print(f"📂 기존 작업 파일({OUTPUT_FILE})을 발견하여 이어서 작업합니다.")
    df_out = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(df_out['problem_id'])
else:
    print(f"📄 새로운 작업 파일({OUTPUT_FILE})을 생성합니다.")
    # 헤더만 있는 빈 CSV 생성
    pd.DataFrame(columns=csv_columns).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

# 작업 범위 슬라이싱
start_idx = max(0, START_IDX - 1)
end_idx = min(len(df_in), END_IDX)
target_problems = df_in.iloc[start_idx:end_idx]

print(f"✅ 총 {len(df_in)}문제 중 {START_IDX}번째 ~ {end_idx}번째 문제(총 {len(target_problems)}개) 분석을 시작합니다.\n")

processed_count = 0

for table_idx, (index, row) in enumerate(target_problems.iterrows(), start=START_IDX):
    prob_id = row['id']
    
    # 이미 분석이 완료된 문제인지 확인 (Skip 로직)
    if prob_id in processed_ids:
        print(f"⏭ [{table_idx}번째] Problem {prob_id} 은(는) 이미 분석이 완료되어 건너뜁니다.")
        continue

    print(f"\n⏳ [{table_idx}번째] Problem {prob_id} 알고리즘 조합 분석 중...")
    step_start_time = time.time()
    
    # 1. LLM에 제공할 텍스트 묶기 (한국어 번역본 대신 원본 영어 question 열만 사용)
    # NaN일 경우 빈 문자열 반환하도록 안전하게 처리
    original_question = str(row.get('question', '')) if pd.notna(row.get('question')) else ""
    problem_context = f"[Original English Problem]\n{original_question}".strip()
    
    # LLM API 호출
    combos = classify_algorithms_with_llm(problem_context)
    
    elapsed_time = time.time() - step_start_time
    
    if combos:
        rows_to_save = []
        
        # LLM이 반환한 여러 조합 배열을 순회하며 CSV 행 데이터 만들기
        for order_idx, combo in enumerate(combos, start=1):
            row_dict = {
                'problem_id': prob_id,
                'algorithm_order': order_idx
            }
            
            true_count = 0
            # 업데이트된 ALGORITHMS 리스트를 훑으면서 매핑
            for algo in ALGORITHMS:
                # LLM이 반환한 딕셔너리에 해당 알고리즘이 있고, 값이 True인지 확인
                is_used = bool(combo.get(algo, False))
                row_dict[algo] = is_used
                if is_used:
                    true_count += 1
                    
            row_dict['count'] = true_count
            rows_to_save.append(row_dict)
        
        # Pandas로 변환 후 실시간 Append 저장
        df_new_rows = pd.DataFrame(rows_to_save, columns=csv_columns)
        df_new_rows.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding='utf-8-sig')
        
        # 콘솔 출력 (사용된 알고리즘 리스트를 로그에 이쁘게 출력)
        print(f"  -> ✅ 분석 완료! (소요 시간: {elapsed_time:.1f}초)")
        for r in rows_to_save:
            used_algos = [a for a in ALGORITHMS if r[a]]
            print(f"     [조합 {r['algorithm_order']}] {len(used_algos)}개 핵심 알고리즘 사용: {', '.join(used_algos)}")
            
        processed_ids.add(prob_id)
        processed_count += 1
    else:
        print(f"  -> ❌ 변환에 실패했습니다. (소요 시간: {elapsed_time:.1f}초)")
    
    # Rate Limit 방어를 위한 대기
    time.sleep(5)

print(f"\n🎉 작업 완료! 이번 실행에서 총 {processed_count}개의 문제에 대한 핵심 알고리즘 조합이 도출/저장되었습니다.")
print(f"📁 결과 저장 파일: {OUTPUT_FILE}")

🚀 [Step 1] 데이터 로드 및 초기화...
📂 기존 작업 파일(problem_algorithms.csv)을 발견하여 이어서 작업합니다.
✅ 총 3544문제 중 1번째 ~ 360번째 문제(총 360개) 분석을 시작합니다.

⏭ [1번째] Problem 2.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [2번째] Problem 11.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [3번째] Problem 19.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [4번째] Problem 41.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [5번째] Problem 51.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [6번째] Problem 53.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [7번째] Problem 58.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [8번째] Problem 59.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [9번째] Problem 64.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [10번째] Problem 73.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [11번째] Problem 78.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [12번째] Problem 89.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [13번째] Problem 92.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [14번째] Problem 96.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [15번째] Problem 100.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [16번째] Problem 106.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [17번째] Problem 113.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [18번째] Problem 116.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [19번째] Problem 128.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [2